# به‌روزرسانی با GPU: OCR، معادله‌ها و سایت (نسخه‌ی 12)

**سلول ۱** (حدود ۱ تا ۲ ساعت؛ Runtime → Change runtime type → T4 GPU):
- متن OCR‌شده‌ی ۴۵ مقاله‌ی اسکن‌شده (همان که روی سایت است) وارد پایگاه کولب می‌شود؛ ۹ مقاله‌ای که PDFشان هیچ متنی ندارد OCR می‌شوند؛
- برای ۴۶ مقاله‌ی ارزیابی فرمول، معادله‌ها با مدل CodeFormula از تصویر صفحه به LaTeX خوانده می‌شوند (GPU)؛
- ۶ مقاله که PDFشان مقاله‌ی دیگری است فقط با چکیده نمایه می‌شوند؛
- پایگاه در Drive ذخیره، PDFها و بخش‌ها در سایت به‌روز و جست‌وجوی سایت دوباره ارزیابی می‌شود.

فهرست مقاله‌ها در `data/rag-optics/*.txt` مخزن است. اگر قطع شد دوباره اجرا کنید؛ از همان‌جا ادامه می‌دهد.

**سلول ۲** (اختیاری، حدود ۱ تا ۲ ساعت): ارزیابی دوباره‌ی استخراج فرمول با مدل qwen3:8b روی همین متن‌های تازه.

In [ ]:
#@title ▶️ ۱. OCR، معادله‌ها، پایگاه و سایت

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_STORAGE_BACKUP_DIR'] = f'{WORK}/storage'
os.environ['MATRAG_CORPUS'] = 'rag-optics'
os.environ['MATRAG_CONVERT_TIMEOUT_S'] = '3600'  # equation decoding is slow on long papers
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "⚠️ No GPU: Runtime → Change runtime type → T4 GPU"
print('Copying the database from Drive...')
!mkdir -p /content/storage && cp -rT "{WORK}/storage" /content/storage

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

!pip install -q "rapidocr>=3" onnxruntime 2>&1 | grep -iE "^error" || true
!cp /content/RAG/data/rag-optics/ocr_papers.txt /content/RAG/data/rag-optics/formula_papers.txt /content/RAG/data/rag-optics/pdf_mismatch.txt "{WORK}/data/rag-optics/"

# Keys: matrag_data/qdrant.env (Qdrant) and matrag_data/cloudflare.env (the site's PDFs) on Drive,
# as KEY=value lines; the Qdrant ones can also come from Colab Secrets.
import os
KEYS_FILE = f'{WORK}/qdrant.env'
for keys_file in (KEYS_FILE, f'{WORK}/cloudflare.env'):
    if os.path.exists(keys_file):
        for line in open(keys_file, encoding='utf-8'):
            key, sep, value = line.strip().partition('=')
            if sep and not key.startswith('#'):
                os.environ[key.strip()] = value.strip().strip('"')
if not os.environ.get('QDRANT_API_KEY'):
    from google.colab import userdata
    for key in ('QDRANT_URL', 'QDRANT_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
assert os.environ.get('QDRANT_URL') and os.environ.get('QDRANT_API_KEY'), f'Qdrant key not found ({KEYS_FILE})'

# The OCR conversions made for the site (same text on both sides), kept in Qdrant.
import base64, pathlib
from matrag import qdrant_store
root = pathlib.Path(open(f'{WORK}/data/rag-optics/corpus_root.txt').read().strip())
qc, offset, got = qdrant_store.client(), None, 0
while True:
    points, offset = qc.scroll('matrag_files', limit=16, offset=offset, with_payload=True)
    for p in points:
        dst = root / '_RAG_processed' / p.payload['path']
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(base64.b64decode(p.payload['data'])); got += 1
    if offset is None:
        break
print(f'{got} OCR conversions copied')
%cd {WORK}
!matrag --corpus rag-optics ingest 2>&1 | grep -vE "Warning|Loading|it/s\]" | tee -a "{WORK}/logs/ingest.log"
if os.environ.get('CLOUDFLARE_API_TOKEN'):
    !matrag --corpus rag-optics export-pdfs /content/RAG/web-pdfs/pdfs 2>&1 | grep -vE "Warning|Loading"
    if not os.path.exists('/content/node/bin/node'):
        !curl -fsSL https://nodejs.org/dist/v22.12.0/node-v22.12.0-linux-x64.tar.xz | tar -xJ -C /content && mv /content/node-v22.12.0-linux-x64 /content/node
    os.environ['PATH'] = '/content/node/bin:' + os.environ['PATH']
    !cd /content/RAG/web-pdfs && npx --yes wrangler@4 deploy 2>&1 | grep -vE "telemetry|^$" | tail -4
!matrag --corpus rag-optics export-qdrant 2>&1 | grep -vE "Warning|Loading"
if os.path.exists(f'{WORK}/data/rag-optics/gold/questions.csv'):
    !matrag --corpus rag-optics evaluate retrieval --backend qdrant 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG
print('✅ Done: https://matrag.ethanjamescarter1995.workers.dev')

In [ ]:
#@title ۲. (اختیاری) ارزیابی دوباره‌ی استخراج فرمول با qwen3:8b
#@markdown نتیجه جدا از ارزیابی قبلی در `results/eval/rag-optics_formula_benchmark_ocr_formulas.jsonl` ذخیره می‌شود.
LIMIT = 80  #@param {type:"integer"}
import subprocess, time
if not os.path.isdir('/content/refractiveindex'):
    !git clone -q --depth 1 https://github.com/polyanskiy/refractiveindex.info-database.git /content/refractiveindex
os.environ['MATRAG_REFERENCE_DIR'] = '/content/refractiveindex/database'
os.environ.update({'MATRAG_LLM_PROVIDER': 'ollama', 'MATRAG_OLLAMA_MODEL': 'qwen3:8b', 'MATRAG_OLLAMA_THINKING': 'false',
                   'OLLAMA_MODELS': '/content/ollama_models'})
if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
    !apt-get install -y -qq zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True)
time.sleep(5)
!ollama pull qwen3:8b 2>&1 | tail -1
%cd {WORK}
!matrag --corpus rag-optics evaluate formulas --run ocr_formulas --limit {LIMIT} 2>&1 | grep -vE "Warning|Loading|it/s\]" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG